# MAIZ_14B2 · Ablación completa del componente sano derivado de PlantVillage · GPU L4

**Entorno: GPU L4.** Pulse **Ejecutar todas**. Tarda unas 3–4 h.

Al rastrear el componente Kaggle_Healthy_paren (709 imágenes sanas) encontramos que **todas** vienen de PlantVillage:
- 360 son copias idénticas;
- 349 son **espejos horizontales**.

El MAIZ_14B retiró solo las copias idénticas. Este notebook retira **todas** las imágenes derivadas de PlantVillage de train y validación:
- **496 en train y 107 en validación.**
- Se identifican con pHash D4 (las 8 rotaciones y espejos).
- Debe coincidir con 496 / 107 / 106, que es exactamente el componente Kaggle_Healthy_paren.

Todo lo demás es igual que en el 14B: EfficientNet-B0, misma receta y selección por validación. Al final hay una evaluación pareada contra el checkpoint público original de la misma semilla.

**Por qué tres semillas:** en el 14B (una sola semilla) el recall de roya en Pandian subió de 0,33 a 0,78. Pero las 5 semillas originales de EfficientNet ya varían entre 0,33 y 0,88, así que con una sola corrida no se puede separar el efecto de la ablación del azar del entrenamiento. Con 3 semillas (17, 42 y 73) se comparan medias pareadas.

| Sesión de Colab | Valor de `JOB_INDICES` | Semilla | Tiempo L4 |
|---|---|---|---|
| A | `[0]` | 17 | ~5,5–6 h |
| B | `[1]` | 42 | ~5,5–6 h |
| C | `[2]` | 73 | ~5,5–6 h |

Si Colab permite tres sesiones L4 a la vez, terminan las tres en unas 6 h. Si no, córralas en dos tandas. Cada sesión evalúa todas las corridas que ya estén completas; la última en terminar produce el ZIP con las tres.

**Reanudación:** igual que el 14B.

**Resultado:** `MAIZ_14B2_Ablacion_Completa_Registros.zip`, para enviármelo.

In [ ]:
import shutil,subprocess
if not shutil.which('nvidia-smi'):raise RuntimeError('Este archivo requiere GPU L4. Selecciónela en Entorno de ejecución > Cambiar tipo.')
p=subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True,timeout=15)
GPU_NAME=p.stdout.strip()
if p.returncode!=0 or 'L4' not in GPU_NAME:raise RuntimeError('Seleccione GPU L4 para este archivo. GPU detectada: '+GPU_NAME)
print('GPU:',GPU_NAME)

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/TesisQ1_Maiz')
if not ROOT.is_dir() or not (ROOT/'01_datasets').is_dir():
    raise FileNotFoundError('No se encuentra el proyecto original TesisQ1_Maiz/01_datasets en Mi unidad. No se creó otra carpeta de datos.')
OUT=ROOT/'REVISION_POST_REVIEW_20261029';OUT.mkdir(exist_ok=True)
print('Proyecto confirmado:',ROOT)
print('Imágenes originales:',ROOT/'01_datasets')

In [ ]:
import json
report_path=OUT/'training_data_preflight.json'
if not report_path.is_file() or json.loads(report_path.read_text()).get('status')!='READY':
    raise RuntimeError('Primero ejecute MAIZ_03A_Preparar_Datos_CPU.ipynb con CPU hasta DATOS LISTOS. Este notebook no prepara datos ni inició entrenamiento.')
print('Preparación de datos encontrada:',report_path)

In [ ]:
import base64,hashlib,io,importlib.util,json,sys,zipfile,traceback
CODE=Path('/content/maiz_revision_code_fix03');CODE.mkdir(exist_ok=True)
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()=='1eef013a977352d893739122b1d9d0db82e97b4043d6ef2a51bfd518ae31a773'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for item in z.infolist():
        p=(CODE/item.filename).resolve();p.relative_to(CODE.resolve());p.write_bytes(z.read(item))
sys.path.insert(0,str(CODE))
for name in ['training_data_preflight','post_review_gpu']:
    spec=importlib.util.spec_from_file_location(name,CODE/(name+'.py'))
    module=importlib.util.module_from_spec(spec);sys.modules[name]=module;spec.loader.exec_module(module)
from post_review_gpu import REVISION,check_executable_sources
print('Versión:',REVISION)
assert len(check_executable_sources(CODE/'e2_original_decoded.py'))==30
print('Archivos finales de ejecución: compilación correcta.')

In [ ]:
import importlib.metadata as metadata
def installed(name):
    try:return metadata.version(name)
    except metadata.PackageNotFoundError:return None
expected={'torch':'2.11.0+cu128','torchvision':'0.26.0+cu128','timm':'1.0.15'}
observed={n:installed(n) for n in expected}
(OUT/'environment_setup_diagnostic.json').write_text(json.dumps({'notebook':'GPU_ONLY','expected':expected,'observed':observed,'gpu':GPU_NAME},indent=2))
print('Versiones:',observed)
missing={n:v for n,v in expected.items() if observed[n]!=v}
if missing:
    loaded=any(n in sys.modules for n in ['torch','torchvision','timm'])
    if 'torch' in missing or 'torchvision' in missing:
        packages=['torch==2.11.0+cu128','torchvision==0.26.0+cu128']
        if installed('torchaudio') is not None:packages.append('torchaudio==2.11.0+cu128')
        subprocess.check_call([sys.executable,'-m','pip','install',*packages,'--index-url','https://download.pytorch.org/whl/cu128'])
    if 'timm' in missing:subprocess.check_call([sys.executable,'-m','pip','install','timm==1.0.15'])
    if loaded:raise RuntimeError('Instalación terminada. Reinicie SOLO la sesión Python, conserve la máquina L4 y ejecute todas de nuevo.')
from post_review_gpu import preflight_versions
environment=preflight_versions(OUT/'environment_preflight.json')

In [ ]:
# Código MAIZ_14B (usa post_review_gpu sin modificarlo)
import base64,hashlib,io,zipfile,importlib.util,subprocess
if importlib.util.find_spec('imagehash') is None:subprocess.run([sys.executable,'-m','pip','install','-q','imagehash==4.3.1'],check=True)
raw14=base64.b64decode('...')
assert hashlib.sha256(raw14).hexdigest()=='2e4a3b9e6a201629b4d810746f2edcf9a951c02b8493e919f9b5cd0cae6d826a'
zipfile.ZipFile(io.BytesIO(raw14)).extractall(CODE)
spec=importlib.util.spec_from_file_location('maiz14b2',CODE/'maiz14b2.py')
ab=importlib.util.module_from_spec(spec);sys.modules['maiz14b2']=ab;spec.loader.exec_module(ab)
src=(CODE/'e2_original_decoded.py').read_text()
for i in range(len(ab.JOBS)):compile(ab.build_source(src,i,str(OUT)),'chk','exec')
print('Versión:',ab.REVISION,'| corridas:',[ab.job_id(i) for i in range(len(ab.JOBS))])

In [ ]:
# 1) Identificación de las imágenes healthy vinculadas a PlantVillage (CPU, pocos minutos)
SHA=ab.identify(ROOT,CODE)
print('Imágenes retiradas de train+validación:',len(SHA))

In [ ]:
# 2) Entrenamiento (cola). Solo EfficientNet-B0
JOB_INDICES=[0]   # Sesión A: [0] · Sesión B: [1] · Sesión C: [2]
try:
    for index in JOB_INDICES:
        ab.train(ROOT,CODE/'e2_original_decoded.py',index,SHA)
except Exception:
    (OUT/'MAIZ14B_GPU_EXECUTION_ERROR.log').write_text(traceback.format_exc());raise

In [ ]:
# 3) Evaluación pareada: checkpoint original público vs. ablación, mismo código
import pandas as pd
res=ab.evaluate(ROOT,CODE)
pd.set_option('display.width',250);display(res.T)

In [ ]:
# 4) Exportar registros (no incluye checkpoints)
ZIP=ab.export(ROOT);print('Registros:',ZIP)
from google.colab import files
files.download(str(ZIP))